 Kaggle Notebook
 │
 ├── %%writefile modular/config.py
 ├── %%writefile modular/data_setup.py
 ├── %%writefile modular/model_setup.py
 ├── %%writefile modular/train.py
 ├── %%writefile modular/save_model.py
 ├── %%writefile modular/run_training.py
 │
 ├── pip install
 ├── run_training.py
 │
 └── zip artifacts
        ↓
     download ZIP
        ↓
     GitHub / Hugging Face
        ↓
     Docker + vLLM

In [1]:
%pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    trl \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/going_modular")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

/kaggle/working/going_modular


In [3]:
%%writefile /kaggle/working/going_modular/config.py

from dataclasses import dataclass


@dataclass
class TrainingConfig:

    # ---------------------------------------------------------
    # Base model
    # ---------------------------------------------------------

    model_id: str = (
        "Joycean0301/llama-3.2-3B-Instruct-Medical"
    )

    # ---------------------------------------------------------
    # Dataset
    # ---------------------------------------------------------

    dataset_path: str = (
        "/kaggle/input/asclepius-5k/asclepius_5k"
    )

    # ---------------------------------------------------------
    # Output
    # ---------------------------------------------------------

    output_dir: str = (
        "/kaggle/working/medical_lora_output"
    )

    final_adapter_dir: str = (
        "/kaggle/working/medical_lora_output/final_adapter"
    )

    # ---------------------------------------------------------
    # Sequence length
    # ---------------------------------------------------------

    max_seq_length: int = 2048

    # ---------------------------------------------------------
    # LoRA
    # ---------------------------------------------------------

    lora_r: int = 16

    lora_alpha: int = 32

    lora_dropout: float = 0.05

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    learning_rate: float = 2e-4

    batch_size: int = 1

    gradient_accumulation_steps: int = 8

    # Short demonstration run.
    #
    # Change this if desired, but this is intentionally
    # NOT a long training experiment.

    max_steps: int = 100

    # Checkpoint frequently so Kaggle interruptions
    # do not destroy the experiment.

    save_steps: int = 25

    logging_steps: int = 5

    save_total_limit: int = 3

    # ---------------------------------------------------------
    # Reproducibility
    # ---------------------------------------------------------

    seed: int = 42


CONFIG = TrainingConfig()

Writing /kaggle/working/going_modular/config.py


In [4]:
max_steps = 150

In [5]:
%%writefile /kaggle/working/going_modular/data_setup.py

import os
from datasets import Dataset, DatasetDict, load_dataset


SYSTEM_PROMPT = (
    "You are a medical language assistant. "
    "Answer the user's clinical instruction accurately "
    "and concisely. "
    "Do not invent information that is not present "
    "in the provided clinical context."
)


def load_medical_dataset(dataset_path):
    """
    Loads arrow splits directly from the given directory path
    and returns a DatasetDict.
    """
    arrow_files = {
        "train": os.path.join(dataset_path, "/kaggle/input/datasets/shehrozeshahzad/medicalai/train.arrow"),
        "validation": os.path.join(dataset_path, "/kaggle/input/datasets/shehrozeshahzad/medicalai/validation.arrow"),
        "test": os.path.join(dataset_path, "/kaggle/input/datasets/shehrozeshahzad/medicalai/validation.arrow"),
    }

    # Verify that all required files exist
    for split, path in arrow_files.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing required file: {path}")

    # Option A: Using load_dataset directly
    dataset = load_dataset("arrow", data_files=arrow_files)

    print("Dataset loaded successfully.")

    for split in ["train", "validation", "test"]:
        print(f"{split}: {len(dataset[split])} examples")

    return dataset


def build_messages(example):

    user_content = (
        "Clinical Note:\n"
        f"{example['clinical_note']}\n\n"
        "Instruction:\n"
        f"{example['instruction']}"
    )

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": example["target"],
        },
    ]


def tokenize_dataset(dataset, tokenizer, max_length):

    def tokenize(example):

        messages = build_messages(example)

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        result = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
        )

        result["labels"] = result["input_ids"].copy()

        return result

    tokenized = {}

    for split in [
        "train",
        "validation",
        "test",
    ]:

        print(f"\nTokenizing {split}...")

        tokenized[split] = dataset[split].map(
            tokenize,
            remove_columns=dataset[split].column_names,
            desc=f"Tokenizing {split}",
        )

    return tokenized

Writing /kaggle/working/going_modular/data_setup.py


In [6]:
%%writefile /kaggle/working/going_modular/model_setup.py

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


def load_tokenizer(model_id):

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if tokenizer.chat_template is None:
        raise RuntimeError(
            "Model tokenizer does not provide "
            "a chat template."
        )

    print("Tokenizer loaded.")

    return tokenizer


def load_qlora_model(model_id):

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required for QLoRA training."
        )

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    bnb_config = BitsAndBytesConfig(

        load_in_4bit=True,

        bnb_4bit_quant_type="nf4",

        bnb_4bit_use_double_quant=True,

        bnb_4bit_compute_dtype=(
            torch.bfloat16
        ),
    )

    model = AutoModelForCausalLM.from_pretrained(

        model_id,

        quantization_config=bnb_config,

        device_map="auto",

        torch_dtype=torch.bfloat16,

        trust_remote_code=True,
    )

    model.config.use_cache = False

    model = prepare_model_for_kbit_training(
        model
    )

    print("4-bit model loaded.")

    return model


def attach_lora(
    model,
    r,
    alpha,
    dropout,
):

    lora_config = LoraConfig(

        r=r,

        lora_alpha=alpha,

        lora_dropout=dropout,

        bias="none",

        task_type="CAUSAL_LM",
    )

    model = get_peft_model(
        model,
        lora_config,
    )

    print("\nLoRA configuration:")

    model.print_trainable_parameters()

    return model

Writing /kaggle/working/going_modular/model_setup.py


In [7]:
%%writefile /kaggle/working/going_modular/train.py

import os
import random

import numpy as np
import torch

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from transformers.trainer_utils import get_last_checkpoint


def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def find_checkpoint(output_dir):

    if not os.path.isdir(output_dir):
        return None

    checkpoint = get_last_checkpoint(
        output_dir
    )

    if checkpoint:

        print(
            f"\nExisting checkpoint found:"
            f"\n{checkpoint}\n"
        )

    return checkpoint


def build_trainer(
    model,
    tokenizer,
    tokenized_dataset,
    config,
):

    # ---------------------------------------------------------
    # Training arguments
    # ---------------------------------------------------------

    training_args = TrainingArguments(

        output_dir=config.output_dir,

        max_steps=config.max_steps,

        per_device_train_batch_size=(
            config.batch_size
        ),

        gradient_accumulation_steps=(
            config.gradient_accumulation_steps
        ),

        learning_rate=config.learning_rate,

        # -----------------------------------------------------
        # Logging
        # -----------------------------------------------------

        logging_steps=config.logging_steps,

        # -----------------------------------------------------
        # Checkpointing
        # -----------------------------------------------------

        save_steps=config.save_steps,

        save_total_limit=config.save_total_limit,

        # -----------------------------------------------------
        # Evaluation
        # -----------------------------------------------------

        eval_steps=config.save_steps,

        # -----------------------------------------------------
        # Precision
        # -----------------------------------------------------

        bf16=True,

        fp16=False,

        # -----------------------------------------------------
        # Memory
        # -----------------------------------------------------

        gradient_checkpointing=True,

        # -----------------------------------------------------
        # Optimizer
        # -----------------------------------------------------

        optim="paged_adamw_8bit",

        weight_decay=0.01,

        # -----------------------------------------------------
        # Reporting
        # -----------------------------------------------------

        report_to="none",

        # -----------------------------------------------------
        # Dataset
        # -----------------------------------------------------

        remove_unused_columns=False,

        # -----------------------------------------------------
        # Reproducibility
        # -----------------------------------------------------

        seed=config.seed,
    )

    # ---------------------------------------------------------
    # Causal language-model collator
    # ---------------------------------------------------------

    data_collator = DataCollatorForLanguageModeling(

        tokenizer=tokenizer,

        mlm=False,
    )

    # ---------------------------------------------------------
    # Trainer
    # ---------------------------------------------------------

    trainer = Trainer(

        model=model,

        args=training_args,

        train_dataset=(
            tokenized_dataset["train"]
        ),

        eval_dataset=(
            tokenized_dataset["validation"]
        ),

        data_collator=data_collator,
    )

    return trainer


def train_model(
    trainer,
    output_dir,
):

    checkpoint = find_checkpoint(
        output_dir
    )

    if checkpoint:

        print(
            "========================================"
        )

        print(
            "RESUMING FROM CHECKPOINT"
        )

        print(
            "========================================"
        )

        result = trainer.train(
            resume_from_checkpoint=checkpoint
        )

    else:

        print(
            "========================================"
        )

        print(
            "STARTING NEW TRAINING RUN"
        )

        print(
            "========================================"
        )

        result = trainer.train()

    return result

Writing /kaggle/working/going_modular/train.py


In [8]:
%%writefile /kaggle/working/going_modular/save_model.py

import json
import os
from datetime import datetime


def save_adapter(
    model,
    tokenizer,
    output_dir,
    config,
):

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    print(
        "\nSaving LoRA adapter..."
    )

    model.save_pretrained(
        output_dir
    )

    tokenizer.save_pretrained(
        output_dir
    )

    metadata = {

        "base_model": config.model_id,

        "method": "QLoRA",

        "dataset": (
            "Asclepius Synthetic "
            "Clinical Notes"
        ),

        "dataset_size": 5000,

        "training_examples": 4000,

        "validation_examples": 500,

        "test_examples": 500,

        "max_steps": config.max_steps,

        "learning_rate": (
            config.learning_rate
        ),

        "lora_r": config.lora_r,

        "lora_alpha": config.lora_alpha,

        "lora_dropout": (
            config.lora_dropout
        ),

        "max_seq_length": (
            config.max_seq_length
        ),

        "created_at": (
            datetime.utcnow()
            .isoformat()
        ),
    }

    with open(
        os.path.join(
            output_dir,
            "training_metadata.json",
        ),
        "w",
    ) as f:

        json.dump(
            metadata,
            f,
            indent=2,
        )

    print(
        f"Adapter saved to:\n{output_dir}"
    )

Writing /kaggle/working/going_modular/save_model.py


In [9]:
%%writefile /kaggle/working/going_modular/run_training.py

import torch

from config import CONFIG

from data_setup import (
    load_medical_dataset,
    tokenize_dataset,
)

from model_setup import (
    load_tokenizer,
    load_qlora_model,
    attach_lora,
)

from train import (
    set_seed,
    build_trainer,
    train_model,
)

from save_model import (
    save_adapter,
)


def main():

    print("=" * 70)

    print(
        "MEDICAL LLM QLoRA TRAINING"
    )

    print("=" * 70)

    print(
        f"Model: {CONFIG.model_id}"
    )

    print(
        f"Max steps: {CONFIG.max_steps}"
    )

    print(
        f"Batch size: {CONFIG.batch_size}"
    )

    print(
        "Gradient accumulation:",
        CONFIG.gradient_accumulation_steps,
    )

    print("=" * 70)

    # ---------------------------------------------------------
    # Seed
    # ---------------------------------------------------------

    set_seed(CONFIG.seed)

    # ---------------------------------------------------------
    # GPU
    # ---------------------------------------------------------

    if not torch.cuda.is_available():

        raise RuntimeError(
            "CUDA GPU is required."
        )

    # ---------------------------------------------------------
    # Dataset
    # ---------------------------------------------------------

    dataset = load_medical_dataset(
        CONFIG.dataset_path
    )

    # ---------------------------------------------------------
    # Tokenizer
    # ---------------------------------------------------------

    tokenizer = load_tokenizer(
        CONFIG.model_id
    )

    # ---------------------------------------------------------
    # Model
    # ---------------------------------------------------------

    model = load_qlora_model(
        CONFIG.model_id
    )

    # ---------------------------------------------------------
    # LoRA
    # ---------------------------------------------------------

    model = attach_lora(

        model,

        CONFIG.lora_r,

        CONFIG.lora_alpha,

        CONFIG.lora_dropout,
    )

    # ---------------------------------------------------------
    # Tokenization
    # ---------------------------------------------------------

    tokenized = tokenize_dataset(

        dataset,

        tokenizer,

        CONFIG.max_seq_length,
    )

    # ---------------------------------------------------------
    # Trainer
    # ---------------------------------------------------------

    trainer = build_trainer(

        model,

        tokenizer,

        tokenized,

        CONFIG,
    )

    # ---------------------------------------------------------
    # Training
    # ---------------------------------------------------------

    result = train_model(

        trainer,

        CONFIG.output_dir,
    )

    print(
        "\nTraining completed."
    )

    print(result)

    # ---------------------------------------------------------
    # Save final adapter
    # ---------------------------------------------------------

    save_adapter(

        trainer.model,

        tokenizer,

        CONFIG.final_adapter_dir,

        CONFIG,
    )

    # ---------------------------------------------------------
    # Save trainer state
    # ---------------------------------------------------------

    trainer.save_state()

    print(
        "\nEverything completed successfully."
    )


if __name__ == "__main__":

    main()

Writing /kaggle/working/going_modular/run_training.py


In [10]:
!cd /kaggle/working/going_modular && python run_training.py

MEDICAL LLM QLoRA TRAINING
Model: Joycean0301/llama-3.2-3B-Instruct-Medical
Max steps: 100
Batch size: 1
Gradient accumulation: 8
Generating train split: 4000 examples [00:00, 15846.50 examples/s]
Generating validation split: 500 examples [00:00, 18046.69 examples/s]
Generating test split: 500 examples [00:00, 63142.50 examples/s]
Dataset loaded successfully.
train: 4000 examples
validation: 500 examples
test: 500 examples
tokenizer_config.json: 55.4kB [00:00, 18.6MB/s]
tokenizer.json: 100%|██████████████████████| 17.2M/17.2M [00:01<00:00, 17.1MB/s]
special_tokens_map.json: 100%|█████████████████| 454/454 [00:00<00:00, 2.74MB/s]
Tokenizer loaded.
GPU: Tesla T4
adapter_config.json: 100%|█████████████████████| 738/738 [00:00<00:00, 5.45MB/s]
config.json: 1.47kB [00:00, 7.07MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters

In [11]:
import shutil
from pathlib import Path

source = "/kaggle/working/medical_lora_output"
destination = "/kaggle/working/medical_lora_artifacts"

shutil.make_archive(
    destination,
    "zip",
    source,
)

print(
    f"Created:\n{destination}.zip"
)

Created:
/kaggle/working/medical_lora_artifacts.zip


In [12]:
import shutil
from pathlib import Path
from IPython.display import FileLink, HTML

# Paths
source_dir = Path("/kaggle/working/medical_lora_output")
zip_name = "/kaggle/working/medical_lora_artifacts"  # shutil.make_archive automatically appends .zip

# 1. Zip the folder
shutil.make_archive(zip_name, "zip", source_dir)

# 2. Generate and display the direct download link
zip_file_path = f"{zip_name}.zip"

if Path(zip_file_path).exists():
    print(f"Archive successfully created at: {zip_file_path}\n")
    # Display HTML download link in the notebook output
    display(
        HTML(
            f'<a href="file/{zip_file_path}" download target="_blank" style="font-size: 18px; font-weight: bold; color: #2088c2;">'
            f"Click here to download {Path(zip_file_path).name}"
            f"</a>"
        )
    )
else:
    print("Failed to create zip file. Check if the source directory exists.")

Archive successfully created at: /kaggle/working/medical_lora_artifacts.zip



In [13]:
from IPython.display import HTML, FileLink
from pathlib import Path

zip_file_path = Path("/kaggle/working/medical_lora_artifacts.zip")

if zip_file_path.exists():
    # Primary Method: Clickable HTML Link
    display(
        HTML(
            f'<a href="file/{zip_file_path}" download target="_blank" '
            f'style="font-size: 18px; font-weight: bold; color: #2088c2; text-decoration: underline;">'
            f"Click here to download {zip_file_path.name}"
            f"</a>"
        )
    )
    # Backup Method: Jupyter FileLink
    display(FileLink(str(zip_file_path)))
else:
    print(f"File not found at {zip_file_path}. Please double-check the path.")

/kaggle/working/medical_lora_artifacts.zip

In [14]:
from IPython.display import FileLink

# Pass ONLY the file name, not the full absolute path
FileLink(r'medical_lora_artifacts.zip')

/kaggle/working/medical_lora_artifacts.zip

In [15]:
from pathlib import Path

final_dir = Path(
    "/kaggle/working/medical_lora_output/final_adapter"
)

for path in sorted(final_dir.iterdir()):
    print(path.name)

README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
tokenizer.json
tokenizer_config.json
training_metadata.json


In [16]:
import json

path = "/kaggle/working/medical_lora_output/final_adapter/adapter_config.json"

with open(path) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "unsloth/llama-3.2-3b-instruct-bnb-4bit",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.20.0",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "q_proj",
    "v_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainable_token_indices": null,
  "use_bdlora": null

In [17]:
import json

path = "/kaggle/working/medical_lora_output/final_adapter/adapter_config.json"

with open(path) as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": null,
  "base_model_name_or_path": "unsloth/llama-3.2-3b-instruct-bnb-4bit",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  "lora_alpha": 32,
  "lora_bias": false,
  "lora_dropout": 0.05,
  "lora_ga_config": null,
  "megatron_config": null,
  "megatron_core": "megatron.core",
  "modules_to_save": null,
  "monteclora_config": null,
  "peft_type": "LORA",
  "peft_version": "0.20.0",
  "qalora_group_size": 16,
  "r": 16,
  "rank_pattern": {},
  "revision": null,
  "target_modules": [
    "q_proj",
    "v_proj"
  ],
  "target_parameters": null,
  "task_type": "CAUSAL_LM",
  "trainable_token_indices": null,
  "use_bdlora": null